# 02 — Data Exploration

**Purpose:** Explore the raw collision CSV to inform schema design decisions for the DuckDB ingest layer.  


## Section 1 — Load and Inspect

Load the raw CSV with all columns read as strings to prevent pandas from silently coercing values  
(e.g. mixed-type ID columns, numeric strings with leading zeros).  
Print the dataset shape, every column name, and the first five rows.

In [7]:
# requirements: pandas
import pandas as pd
from pathlib import Path

RAW_CSV = Path("../data/raw/Traffic_Collision_Data.csv")

df = pd.read_csv(RAW_CSV, dtype=str, encoding="utf-8")

print("Shape:", df.shape)
print(f"\nColumn count: {len(df.columns)}")
print("\nAll columns: ")
print(df.columns.tolist())

print("\nFirst 5 rows:")
df.head()

Shape: (94406, 28)

Column count: 28

All columns: 
['X', 'Y', 'X_Coordinate', 'Y_Coordinate', 'ID', 'Geo_ID', 'Accident_Year', 'Accident_Date', 'Location', 'Classification_Of_Accident', 'Initial_Impact_Type', 'Road_1_Surface_Condition', 'Environment_Condition_1', 'Light', 'Traffic_Control', 'num_of_vehicles', 'num_of_pedestrians', 'num_of_bicycles', 'num_of_motorcycles', 'Max_injury', 'num_of_injuries', 'num_of_minimal', 'num_of_minor', 'num_of_major', 'num_of_fatal', 'Lat', 'Long', 'ObjectId']

First 5 rows:


,X,Y,X_Coordinate,Y_Coordinate,ID,Geo_ID,Accident_Year,Accident_Date,Location,Classification_Of_Accident,...,num_of_motorcycles,Max_injury,num_of_injuries,num_of_minimal,num_of_minor,num_of_major,num_of_fatal,Lat,Long,ObjectId
0,-8449906.52536107,5674411.93776853,351293.8071,5021841.11,2017--101,__3Z00RNB,2017,1/4/2017,MARCH RD btwn 280 S OF CARLING AVE/STATION RD ...,02 - Non-fatal injury,...,NaN,Minimal,1,1,NaN,NaN,NaN,45.3349777290216,-75.9068018111313,1
1,-8432340.77442454,5664915.09787958,363723.6849,5015276.436,2017--201,0004726,2017,1/5/2017,GREENBANK RD @ BERRIGAN DR/WESSEX RD (0004726),03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.2749753021556,-75.7490059856981,2
2,-8427552.17562677,5687484.42888596,366942.7019,5031143.575,2017--801,0006357,2017,1/23/2017,ALBERT ST @ BAY ST (0006357),02 - Non-fatal injury,...,NaN,Minimal,1,1,NaN,NaN,NaN,45.4174677673002,-75.7059892708025,3
3,-8423617.92548895,5682501.10126153,369744.7181,5027678.543,2017--102,0001586,2017,1/4/2017,KILBORN AVE/KILBORN PL @ LAMIRA ST (0001586),03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.3860361640136,-75.670647300499,4
4,-8421027.7190098,5632923.04346668,371935.0538,4992842.24,2017--103,__3Z09V2,2017,1/4/2017,DONNELLY DR btwn FAIRMILE RD & FOURTH LINE RD ...,03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.0723774700594,-75.6473790798065,5


## Section 2 — Null Profile

Quantify missing data across every column.  
The first cell produces a full null summary sorted by null percentage descending.  

The third cell confirms that `Max_injury` nulls align with property-damage-only rows.

In [8]:
# Targeted null and value profile for the numeric count columns only
count_cols = [
    "num_of_vehicles",
    "num_of_pedestrians",
    "num_of_bicycles",
    "num_of_motorcycles",
    "num_of_injuries",
    "num_of_minimal",
    "num_of_minor",
    "num_of_major",
    "num_of_fatal",
]

# Cast to numeric — required because the CSV was loaded with dtype=str
numeric = df[count_cols].apply(pd.to_numeric, errors="coerce")

null_profile = pd.DataFrame({
    "non_null_count": numeric.notna().sum(),
    "null_count":     numeric.isna().sum(),
    "null_pct":       (numeric.isna().sum() / len(df) * 100).round(2),
    "min":            numeric.min(),
    "max":            numeric.max(),
    "mean":           numeric.mean().round(3),
})

null_profile


,non_null_count,null_count,null_pct,min,max,mean
num_of_vehicles,94238,168,0.18,1.0,25.0,1.866
num_of_pedestrians,1831,92575,98.06,1.0,3.0,1.039
num_of_bicycles,1665,92741,98.24,1.0,3.0,1.010
num_of_motorcycles,791,93615,99.16,1.0,3.0,1.015
num_of_injuries,15223,79183,83.87,1.0,38.0,1.293
num_of_minimal,6590,87816,93.02,1.0,11.0,1.208
num_of_minor,8656,85750,90.83,1.0,10.0,1.220
num_of_major,817,93589,99.13,1.0,14.0,1.113
num_of_fatal,169,94237,99.82,1.0,3.0,1.065


In [9]:
# For injury-classified rows, num_of_vehicles and num_of_injuries should never be null.
# The sparse columns (pedestrians, bicycles, motorcycles) being null 
# is expected —  it just means that road user type wasn't involved.

# Only these two columns are required on every injury record
core_cols = ["num_of_vehicles", "num_of_injuries"]

# Match only Fatal and Non-fatal injury rows — P.D. only and Non-reportable are excluded
is_injury = df["Classification_Of_Accident"].str.contains(
    r"Fatal injury|Non-fatal injury", na=False
)
# Flag rows where either core column is missing despite being an injury event
core_null = df[core_cols].isna().any(axis=1)
suspect_rows = df[is_injury & core_null]
print(f"Injury rows missing num_of_vehicles or num_of_injuries: {len(suspect_rows):,}")

if len(suspect_rows) > 0:
    show_cols = [c for c in ["ID", "Classification_Of_Accident", "Max_injury"] + core_cols if c in df.columns]
    display(suspect_rows[show_cols])


Injury rows missing num_of_vehicles or num_of_injuries: 9


,ID,Classification_Of_Accident,Max_injury,num_of_vehicles,num_of_injuries
68237,2022--68922,02 - Non-fatal injury,NaN,NaN,NaN
72545,2022--72715,02 - Non-fatal injury,NaN,2,NaN
77137,2024--100261,01 - Fatal injury,NaN,NaN,NaN
79133,2024--101446,02 - Non-fatal injury,NaN,NaN,NaN
79240,2024--101539,02 - Non-fatal injury,NaN,NaN,NaN
79321,2024--101825,02 - Non-fatal injury,NaN,NaN,NaN
79489,2024--102106,02 - Non-fatal injury,NaN,NaN,NaN
79587,2024--102010,02 - Non-fatal injury,Minor,NaN,1
79804,2024--102237,02 - Non-fatal injury,NaN,NaN,NaN


## Section 3 — Categorical Value Audit

Run `value_counts(dropna=False)` on each key categorical column individually.  
The goal is to document the full set of observed values (including nulls and code prefixes)  
before any cleaning is applied, so schema decisions are grounded in what the data actually contains.

In [10]:
# Column: Classification_Of_Accident
df["Classification_Of_Accident"].value_counts(dropna=False)

Classification_Of_Accident
03 - P.D. only           76671
02 - Non-fatal injury    15002
04 - Non-reportable       2563
01 - Fatal injury          170
Name: count, dtype: int64

In [11]:
# Column: Initial_Impact_Type
df["Initial_Impact_Type"].value_counts(dropna=False)

Initial_Impact_Type
03 - Rear end                  31720
07 - SMV other                 14197
04 - Sideswipe                 13616
02 - Angle                     11825
05 - Turning movement          11164
06 - SMV unattended vehicle     8206
99 - Other                      2387
01 - Approaching                1279
NaN                               12
Name: count, dtype: int64

In [12]:
# Column: Light
df["Light"].value_counts(dropna=False)

Light
01 - Daylight    63755
07 - Dark        21207
05 - Dusk         4310
00 - Unknown      2714
03 - Dawn         2365
99 - Other          41
NaN                 14
Name: count, dtype: int64

In [13]:
# Column: Traffic_Control
df["Traffic_Control"].value_counts(dropna=False)

Traffic_Control
10 - No control            43700
01 - Traffic signal        38250
02 - Stop sign             10185
11 - Roundabout             1309
03 - Yield sign              615
12 - IPS                     130
99 - Other                    81
13 - MPS                      42
04 - Ped. crossover           32
NaN                           30
08 - Traffic gate             12
09 - Traffic controller       10
07 - School bus                6
05 - Police control            3
06 - School guard              1
Name: count, dtype: int64

In [14]:
# Column: Road_1_Surface_Condition
df["Road_1_Surface_Condition"].value_counts(dropna=False)

Road_1_Surface_Condition
01 - Dry                     64171
02 - Wet                     15654
03 - Loose snow               5430
06 - Ice                      3440
04 - Slush                    3056
05 - Packed snow              2366
00 - Unknown                   101
99 - Other                      85
08 - Loose sand or gravel       80
07 - Mud                        12
09 - Spilled liquid             10
NaN                              1
Name: count, dtype: int64

In [15]:
# Column: Environment_Condition_1
df["Environment_Condition_1"].value_counts(dropna=False)

Environment_Condition_1
01 - Clear                     74472
03 - Snow                       9198
02 - Rain                       8261
04 - Freezing Rain              1291
05 - Drifting Snow               439
07 - Fog, mist, smoke, dust      280
99 - Other                       184
06 - Strong wind                 147
00 - Unknown                     121
NaN                               13
Name: count, dtype: int64

In [16]:
# Column: Max_injury
df["Max_injury"].value_counts(dropna=False)

Max_injury
NaN        79241
Minor       8450
Minimal     5751
Major        795
Fatal        169
Name: count, dtype: int64

In [17]:
# Column: Accident_Year
df["Accident_Year"].value_counts(dropna=False).sort_index()

Accident_Year
2017    14424
2018    14556
2019    16465
2020    10067
2021     9748
2022    12490
2024    16656
Name: count, dtype: int64

## Section 4 — Coordinate Quality Check

Check for nulls in `Lat` and `Long`, print their min/max, and flag rows outside Ottawa's  
approximate bounding box (lat 44.9–45.7, lon -76.4 to -75.2).  
Outliers may represent data entry errors, placeholder values (e.g. 0.0), or collisions  
recorded with incorrect coordinates — they should be investigated before geospatial analysis.

In [ ]:
# Cast coordinate columns to float for numeric operations
lat = pd.to_numeric(df["Lat"], errors="coerce")
lon = pd.to_numeric(df["Long"], errors="coerce")

lat_null = lat.isna().sum()
lon_null = lon.isna().sum()
print(f"Lat nulls: {lat_null:,}    Long nulls: {lon_null:,}")
print(f"Lat  — min: {lat.min():.6f}   max: {lat.max():.6f}")
print(f"Long — min: {lon.min():.6f}  max: {lon.max():.6f}")

LAT_MIN, LAT_MAX = 44.9, 45.7
LON_MIN, LON_MAX = -76.4, -75.2

outside_bbox = (
    lat.notna() & lon.notna() &
    ~(lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))
)

flagged_count = outside_bbox.sum()
print(f"\nRows outside Ottawa bounding box: {flagged_count:,}")

if flagged_count > 0:
    flagged = df[outside_bbox].copy()
    flagged["_Lat"] = lat[outside_bbox]
    flagged["_Long"] = lon[outside_bbox]
    show_cols = [c for c in ["ID", "Accident_Date", "Location", "_Lat", "_Long"] if c in flagged.columns]
    print("\nSample flagged rows (up to 20):")
    flagged[show_cols].head(20)

Lat nulls: 0    Long nulls: 0
Lat  — min: 0.000000   max: 45.524921
Long — min: -79.237290  max: -75.261583

Rows outside Ottawa bounding box: 72

Sample flagged rows (up to 20):


: 